#### THIS SCRIPT WAS USED TO FINE TUNE MINISTRAL 8B MENTAL AND MINISTRAL 8B MAPPY.

It was intended to run on google colab due to the GPU requirement.

DO NOT TRY TO RUN THIS!

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls


!pip install -e .[torch,bitsandbytes]
!pip install -U  mistral-common sentencepiece bitsandbytes

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 651, done.
remote: Counting objects: 100% (651/651), done.
remote: Compressing objects: 100% (484/484), done.
remote: Total 651 (delta 159), reused 423 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (651/651), 5.28 MiB | 11.11 MiB/s, done.
Resolving deltas: 100% (159/159), done.
/content/LLaMA-Factory
assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=lla

In [ ]:
%ls

assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/


In [ ]:
import shutil, os

# Unzip from Drive to current directory
shutil.unpack_archive(
    "/content/drive/MyDrive/ministral_lora.zip",
    "ministral_lora"
)
print("Done!")

Done!


In [ ]:
# ============================================================
# Fix JSON parsing bug (Python 3.12)
# ============================================================

from pathlib import Path

parser_file = Path("src/llamafactory/hparams/parser.py")
text = parser_file.read_text()

fixed = text.replace(
    "json.load(Path(sys.argv[1]).absolute())",
    "json.load(open(Path(sys.argv[1]).absolute(), 'r'))"
)

parser_file.write_text(fixed)

print("Patched parser.py")

Patched parser.py


In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

In [ ]:
from google.colab import userdata
import os

In [ ]:
import json
from pathlib import Path

dataset_info_path = Path("./data/dataset_info.json")

# Load existing info or start fresh
if dataset_info_path.exists():
    dataset_info = json.loads(dataset_info_path.read_text())
else:
    dataset_info = {}
print(dataset_info)
# Register your dataset
dataset_info["custom_dr_mappy"]= {
        "file_name": str("custom_dr_mappy.json"),
        "formatting": "alpaca",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output"
        }
    }
# Save back
dataset_info_path.write_text(json.dumps(dataset_info, indent=2))

print("Registered custom dataset in dataset_info.json")

{'identity': {'file_name': 'identity.json'}, 'alpaca_en_demo': {'file_name': 'alpaca_en_demo.json'}, 'alpaca_zh_demo': {'file_name': 'alpaca_zh_demo.json'}, 'glaive_toolcall_en_demo': {'file_name': 'glaive_toolcall_en_demo.json', 'formatting': 'sharegpt', 'columns': {'messages': 'conversations', 'tools': 'tools'}}, 'glaive_toolcall_zh_demo': {'file_name': 'glaive_toolcall_zh_demo.json', 'formatting': 'sharegpt', 'columns': {'messages': 'conversations', 'tools': 'tools'}}, 'mllm_demo': {'file_name': 'mllm_demo.json', 'formatting': 'sharegpt', 'columns': {'messages': 'messages', 'images': 'images'}, 'tags': {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant'}}, 'mllm_audio_demo': {'file_name': 'mllm_audio_demo.json', 'formatting': 'sharegpt', 'columns': {'messages': 'messages', 'audios': 'audios'}, 'tags': {'role_tag': 'role', 'content_tag': 'content', 'user_tag': 'user', 'assistant_tag': 'assistant'}}, 'mllm_video_demo': {'file_name': 'mllm_vi

In [ ]:
from datasets import Dataset
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/dmrs_defenses.csv"
df = pd.read_csv(CSV_PATH).fillna("")

SYSTEM_PROMPT = (
    "You are a mental health analysis assistant trained to analyze mental state based on text posts. "
    "Evaluate the text post provided by the user, answer their question, "
    "and then provide an evidence-based explanation in the voice of the eccentric 'Dr. Dennis Mappy,'"
     "a relaxed, fun-loving, and wise doctor of psychology whom specializes in explaining his logic in a fun way that is still simple and informative."
    "You may ONLY use regular ascii characters, no emojis.  "
)

dataset = Dataset.from_pandas(pd.DataFrame({
    "instruction": [SYSTEM_PROMPT] * len(df),
    "input": df["query"].astype(str).str.strip(),
    "output": df["output"].astype(str).str.strip(),
}), preserve_index=False)

dataset.to_json("./data/custom_dr_mappy.json")

Creating json from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

7081995

In [ ]:
# ============================================================
# Add Ministral support
# ============================================================
'''
from pathlib import Path

# Register model
constants_path = Path("src/llamafactory/extras/constants.py")
text = constants_path.read_text()

if "ministral" not in text:
  print("Patching")
  text += "\nMODEL_GROUPS['ministral'] = ['mistralai/Ministral-8B-Instruct-2410']\n"
  constants_path.write_text(text)

# Register template
template_path = Path("src/llamafactory/data/template.py")
text = template_path.read_text()

if "ministral" not in text:
    text += """

register_template(
    name="ministral",
    format_user="<s>[INST]{content}[/INST]",
    format_assistant="{content}</s>",
    sep=""
)
"""
    template_path.write_text(text)

print("Ministral patched")'''

'\nfrom pathlib import Path\n\n# Register model\nconstants_path = Path("src/llamafactory/extras/constants.py")\ntext = constants_path.read_text()\n\nif "ministral" not in text:\n  print("Patching")\n  text += "\nMODEL_GROUPS[\'ministral\'] = [\'mistralai/Ministral-8B-Instruct-2410\']\n"\n  constants_path.write_text(text)\n\n# Register template\ntemplate_path = Path("src/llamafactory/data/template.py")\ntext = template_path.read_text()\n\nif "ministral" not in text:\n    text += """\n\nregister_template(\n    name="ministral",\n    format_user="<s>[INST]{content}[/INST]",\n    format_assistant="{content}</s>",\n    sep=""\n)\n"""\n    template_path.write_text(text)\n\nprint("Ministral patched")'

In [ ]:
'''
#This was the config used to train the old model.

import json

args = {
  "stage": "sft",
  "do_train": True,

  "model_name_or_path": "mistralai/Ministral-8B-Instruct-2410",

  "dataset": "custom",
  "template": "mistral",

  "finetuning_type": "lora",
  "lora_target": "all",
  "lora_rank": 16,
  "lora_alpha": 32,
  "output_dir": "ministral_lora",
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.03,  # optional, typical warmup fraction
  "per_device_train_batch_size": 8 ,
  "gradient_accumulation_steps": 4,

  "learning_rate": 0.0002,
  "num_train_epochs":3,
  #"max_steps":5,

  "fp16": True,
  #"quantization_bit": 8,
  #"quantization_type": "fp8",

  "cutoff_len": 65536,

  "logging_steps": 100,
  "save_steps": 500,
  "overwrite_output_dir": True,

  "report_to": "none"
  }

json.dump( args, open("train_ministral.json", "w"), indent=2)
# !llamafactory-cli train train_ministral.json'''

'\nimport json\n\nargs = {\n  "stage": "sft",\n  "do_train": True,\n\n  "model_name_or_path": "CrosswaveOmega/ministral8b-mental-lora-unquantized",\n\n  "dataset": "custom",\n  "template": "mistral",\n\n  "finetuning_type": "lora",\n  "lora_target": "all",\n  "lora_rank": 16,\n  "lora_alpha": 32,\n  "output_dir": "ministral_lora",\n  "lr_scheduler_type": "cosine",\n  "warmup_ratio": 0.03,  # optional, typical warmup fraction\n  "per_device_train_batch_size": 8 ,\n  "gradient_accumulation_steps": 4,\n\n  "learning_rate": 0.0002,\n  "num_train_epochs":3,\n  #"max_steps":5,\n\n  "fp16": True,\n  #"quantization_bit": 8,\n  #"quantization_type": "fp8",\n\n  "cutoff_len": 65536,\n\n  "logging_steps": 100,\n  "save_steps": 500,\n  "overwrite_output_dir": True,\n\n  "report_to": "none"\n  }\n\njson.dump( args, open("train_ministral.json", "w"), indent=2)\n# !llamafactory-cli train train_ministral.json'

In [ ]:
import json

args = {
  "stage": "sft",
  "do_train": True,

  "model_name_or_path": "CrosswaveOmega/ministral8b-mental-lora-unquantized",

  "dataset": "custom_dr_mappy",
  "template": "mistral",

  "finetuning_type": "lora",
  "lora_target": "all",
  "lora_rank": 8,
  "lora_alpha": 16,
  "output_dir": "ministral_lora_drmappy",
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.03,  # optional, typical warmup fraction
  "per_device_train_batch_size": 1 ,
  "gradient_accumulation_steps": 8,

  "learning_rate":  1e-4,
  "num_train_epochs":10,
  #"max_steps":5,

  "fp16": True,
  #"quantization_bit": 8,
  #"quantization_type": "fp8",

  "cutoff_len": 65536,

  "logging_steps": 50,
  "save_steps": 500,
  "overwrite_output_dir": True,

  "report_to": "none"
  }

json.dump( args, open("train_ministral.json", "w"), indent=2)
!llamafactory-cli train train_ministral.json

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-04-08 14:57:52] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:667] 2026-04-08 14:57:52,837 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--CrosswaveOmega--ministral8b-mental-lora-unquantized/snapshots/d36aec735d52cbf139522bb19122d7ac1dade4e0/config.json
[INFO|configuration_auto.py:1378] 2026-04-08 14:57:52,837 >> Detected mistral model with layer_types, treating as ministral for alternating attention compatibility. 
[INFO|configuration_utils.py:739] 2026-04-08 14:57:52,839 >> Model config MinistralConfig {
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer

In [ ]:
from google.colab import userdata


In [ ]:
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
import json
merge_args = {
    "model_name_or_path": "CrosswaveOmega/ministral8b-mental-lora-unquantized",
    "adapter_name_or_path": "ministral_lora_drmappy",
    "template": "mistral",
    "finetuning_type": "lora",
    "export_dir": "merged_ministral_drmappy",
    "export_size": 5,

    "export_hub_model_id": "CrosswaveOmega/ministral8b-drmappy-mental-lora-unquantized",
    "export_device": "auto",
}

json.dump(merge_args, open("merge.json", "w"), indent=2)
!llamafactory-cli export merge.json

Streaming output truncated to the last 5000 lines.

  ...0001-of-00004.safetensors:  78% 3.88G/4.95G [00:59<00:16, 63.9MB/s]




Processing Files (0 / 4)      :  51% 8.21G/16.0G [01:01<01:04, 122MB/s,  131MB/s  ]
New Data Upload               :  72% 6.10G/8.52G [01:01<00:19, 122MB/s,  131MB/s  ]

  ...0004-of-00004.safetensors:  88% 1.06G/1.20G [01:00<00:07, 17.6MB/s]


  ...0002-of-00004.safetensors:  35% 1.71G/4.92G [00:59<01:52, 28.5MB/s]



  ...0001-of-00004.safetensors:  79% 3.89G/4.95G [00:59<00:16, 63.8MB/s]




Processing Files (0 / 4)      :  51% 8.23G/16.0G [01:01<01:03, 122MB/s,  130MB/s  ]
New Data Upload               :  72% 6.12G/8.52G [01:01<00:19, 122MB/s,  130MB/s  ]

  ...0004-of-00004.safetensors:  88% 1.06G/1.20G [01:00<00:07, 17.6MB/s]


  ...0002-of-00004.safetensors:  35% 1.72G/4.92G [01:00<01:51, 28.6MB/s]



  ...0001-of-00004.safetensors:  79% 3.90G/4.95G [01:00<00:16, 63.7MB/s]




Processing Files (0 / 4)      :  52% 8.27G/16.0G [01:01<00:57, 136MB/s,  131M

In [ ]:
'''import shutil, os

# Remove checkpoint folders in-place
lora_dir = "ministral_lora_drmappy"
for item in os.listdir(lora_dir):
    if item.startswith("checkpoint-"):
        shutil.rmtree(os.path.join(lora_dir, item))
        print(f"Removed {item}")

# Zip and save to Drive
shutil.make_archive(
    f"/content/drive/MyDrive/{lora_dir}",  # output path (no .zip needed)
    "zip",
    lora_dir
)
print("Saved to Google Drive!")'''

'import shutil, os\n\n# Remove checkpoint folders in-place\nlora_dir = "ministral_lora_drmappy"\nfor item in os.listdir(lora_dir):\n    if item.startswith("checkpoint-"):\n        shutil.rmtree(os.path.join(lora_dir, item))\n        print(f"Removed {item}")\n\n# Zip and save to Drive\nshutil.make_archive(\n    f"/content/drive/MyDrive/{lora_dir}",  # output path (no .zip needed)\n    "zip",\n    lora_dir\n)\nprint("Saved to Google Drive!")'